In [5]:
import re
import pandas as pd
import os
from pathlib import Path

In [6]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [7]:
# Fuente Symbol -> U+F000 + código. Tabla completa de lo que aparece en enunciados.
SYMBOL = {
    # operadores y relaciones
    0x20: " ", 0x21: "!", 0x22: "∀", 0x23: "#", 0x24: "∃", 0x25: "%", 0x26: "&",
    0x28: "(", 0x29: ")", 0x2A: "*", 0x2B: "+", 0x2C: ",", 0x2D: "-", 0x2E: ".",
    0x2F: "/", 0x3A: ":", 0x3B: ";", 0x3C: "<", 0x3D: "=", 0x3E: ">", 0x40: "≅",
    0x5B: "[", 0x5D: "]", 0x5E: "⊥", 0x7B: "{", 0x7C: "|", 0x7D: "}", 0x7E: "~",
    0xA2: "'", 0xA3: "≤", 0xA4: "/", 0xA5: "∞", 0xA6: "f", 0xAB: "↔", 0xAC: "←",
    0xAD: "↑", 0xAE: "→", 0xAF: "↓", 0xB0: "°", 0xB1: "±", 0xB2: "''", 0xB3: "≥",
    0xB4: "×", 0xB5: "∝", 0xB6: "∂", 0xB7: "·", 0xB8: "÷", 0xB9: "≠", 0xBA: "≡",
    0xBB: "≈", 0xBC: "...", 0xC6: "∅", 0xC7: "∩", 0xC8: "∪", 0xCC: "⊂", 0xCE: "∈",
    0xCF: "∉", 0xD0: "∠", 0xD1: "∇", 0xD5: "∏", 0xD6: "√", 0xD7: "·", 0xDB: "⇔",
    0xDC: "⇐", 0xDE: "⇒", 0xE5: "∑", 0xF2: "∫",
    # griegas mayúsculas y minúsculas
    0x44: "Δ", 0x46: "Φ", 0x47: "Γ", 0x4C: "Λ", 0x50: "Π", 0x51: "Θ", 0x53: "Σ",
    0x57: "Ω", 0x59: "Ψ", 0x61: "α", 0x62: "β", 0x63: "χ", 0x64: "δ", 0x65: "ε",
    0x66: "φ", 0x67: "γ", 0x68: "η", 0x69: "ι", 0x6A: "ϕ", 0x6B: "κ", 0x6C: "λ",
    0x6D: "μ", 0x6E: "ν", 0x70: "π", 0x71: "θ", 0x72: "ρ", 0x73: "σ", 0x74: "τ",
    0x75: "υ", 0x77: "ω", 0x78: "ξ", 0x79: "ψ", 0x7A: "ζ",
    # trozos de delimitadores grandes: se deja solo la pieza superior
    0xE6: "(", 0xE7: "", 0xE8: "",            # paréntesis izquierdo
    0xF6: ")", 0xF7: "", 0xF8: "",            # paréntesis derecho
    0xE9: "[", 0xEA: "", 0xEB: "",            # corchete izquierdo
    0xF9: "]", 0xFA: "", 0xFB: "",            # corchete derecho
    0xEC: "{", 0xED: "", 0xEE: "", 0xEF: "",  # llave izquierda (sistemas, funciones a trozos)
    0xFC: "}", 0xFD: "", 0xFE: "",            # llave derecha
    0xF3: "∫", 0xF4: "", 0xF5: "",            # integral grande
}
TABLA = {0xF000 + k: v for k, v in SYMBOL.items()}
TABLA.update({ord("−"): "-", ord("―"): "-", ord("’"): "'"})

IZQ = "\uf0e6\uf0e7\uf0e8"   # paréntesis grandes de matriz
DER = "\uf0f6\uf0f7\uf0f8"

In [8]:
def limpiar(t: str, matrices: bool = False) -> str:
    if matrices:
        t = re.sub(f"[{IZQ}][^{DER}]*[{DER}]+", " [matriz] ", t)
    t = t.translate(TABLA)

    if matrices:  # notación matricial: solo tiene sentido en ese tema
        t = re.sub(r"\bI\s*\n\s*(\d)\b", r"I_\1", t)
        t = re.sub(r"\b([A-Z]) ?- ?1\b(?! *[0-9])", r"\1^(-1)", t)
        t = re.sub(r"\b([A-Z]) ?([2-9])\b", r"\1^\2", t)
        t = re.sub(r"\b([A-Z]) t\b", r"\1^t", t)
        t = re.sub(r"\) t\b", ")^t", t)
        t = re.sub(r"\b([A-Z])t\b", r"\1^t", t)
        t = re.sub(r"([A-Z])-1\b(?![\d.,]\d)", r"\1^(-1)", t)
        t = re.sub(r"\)(\d)\b", r")^\1", t)

    # Comunes a todos los temas
    t = re.sub(r"\b(de|la|matriz)([A-Z(])", r"\1 \2", t)
    t = re.sub(r"\b([a-e]) \)", r"\1)", t)
    t = re.sub(r"\(\s*\n", "(", t)
    t = re.sub(r"\n\s*\)", ")", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r" *\n *", "\n", t)
    t = re.sub(r"\n(?![a-e]\) )", " ", t)
    t = re.sub(r" (?=[a-e]\) [A-ZÁÉÍÓÚ¿(])", "\n", t)
    t = re.sub(r" {2,}", " ", t)
    t = re.sub(r" ([.,;:])", r"\1", t)

    if matrices:
        t = re.sub(r"- (\d)", r"-\1", t)
        TOK = r"(?:(?!y\b|e\b|o\b)[-\w]{1,4})"
        for _ in range(3):
            t = re.sub(rf"\[matriz\](?: {TOK})* \[matriz\]", "[matriz]", t)
            t = re.sub(rf"= ?(?:{TOK} )+\[matriz\]", "= [matriz]", t)
        t = re.sub(r"= ?-?\w{1,3}(?: -?\w{1,3})+(?= [A-Z] ?=| y |,|\.|$)",
                   lambda m: m.group(0) if re.search(r"[a-zλ]{2}", m.group(0)) else "= [matriz]", t)
        t = re.sub(r"^\s*(?:\[matriz\] ?)+(?=[A-ZÁÉÍÓÚ¿a-e])", "", t)
        t = re.sub(r"(?<=[A-Za-záéíóúñ,.]) \[matriz\]", "", t)
        t = re.sub(r" {2,}", " ", t)
    return t.strip()

In [9]:
def glifos_pendientes(serie: pd.Series) -> pd.Series:
    """Cuenta los caracteres de uso privado que quedan sin traducir."""
    return serie.str.findall("[\ue000-\uf8ff]").explode().dropna().map(
        lambda c: f"U+{ord(c):04X}").value_counts()

In [42]:
df = pd.read_csv(root / "csv/ejercicios_clasificados.csv")

In [50]:
df.head(2)

,id,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
0,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,Discuta la veracidad de las siguientes afirmac...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
1,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,Dos conductores rectilíneos muy largos se disp...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...


In [61]:
df_mates = df[(df.asignatura.str.contains("Mates")) & (df.tema == "matrices y determinantes")]

In [62]:
df_mates["enunciado_limpio"] = df_mates.enunciado.apply(lambda st : limpiar(st, matrices = True))

In [63]:
df_mates.loc[:, "enunciado_limpio"] = df_mates.enunciado

In [64]:
df_mates.sample()

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen,enunciado_limpio
id,,,,,,,,,,,
mates_ii_matrices_y_determinantes_2018_reserva_1_ejercicio_3_opcion_b,Mates II,2018,Reserva 1,"Ejercicio 3, opción B",matrices y determinantes,ecuación matricial,Considera las matrices A= 0 0 -1; B = 1 0 1; C...,MATEMÁTICAS II. 2018. RESERVA 1. EJERCICIO 3. ...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,Considera las matrices A= 0 0 -1; B = 1 0 1; C...


In [66]:
#df = df.set_index("id")
df.update(df_mates.enunciado)

In [ ]:
# df["enunciado_limpio"] = df.enunciado.apply(lambda st : limpiar(st, matrices = True))

In [ ]:
# print(glifos_pendientes(df["enunciado_limpio"]))


Series([], Name: count, dtype: int64)


In [37]:
for enun in df.sample(5).enunciado_limpio:
    print(enun)

Sean A, B, X, Y matrices invertibles que verifican: A·X = B y B·Y = A.
a) (1 punto) Compruebe que Y^(-1) = X.
b) (1'5 puntos) Para A= [matriz] y B= [matriz], halle X e Y.
Se consideran las matrices A= [matriz] B= [matriz] C = [matriz]
a) ¿Tiene inversa la matriz A·B-C? Justifique la respuesta y, en caso afirmativo, calcule (A·B-C)-1.
b) Resuelva la ecuación matricial A·B·X -C·X =C^t.
Se consideran las matrices M = [matriz] N = [matriz] V = [matriz], siendo a un número real
a) Halle el valor de a para que se verifique que M^t · V = ( 5 1 5 )^t.
b) Calcule M^(-1) y resuelva la ecuación matricial X · M - I_3 = N.
c) Razone si las operaciones 2 · V · N^t y (N + M^t)·V se pueden realizar y, en aquellos casos en que sea posible, indique la dimensión de la matriz resultante.
Sea la matriz A= [matriz]
a) ¿Para qué valores del parámetro k no existe la inversa de la matriz A?. Justifica la respuesta.
b) Para k =0, resuelve la ecuación matricial (X +I)·A= A^t, donde I denota la matriz identidad y

In [71]:
df[df.asignatura == "Mates II"].sample(10).enunciado

id
mates_ii_integrales_2008_reserva_1_ejercicio_1_opcion_b                     Sea f :  la función definida por f(x)ax3 bx...
mates_ii_integrales_2009_septiembre_ejercicio_2_opcion_b                    x\nSea f la función definida por f(x) .\n49x...
mates_ii_sistemas_de_ecuaciones_lineales_2017_junio_ejercicio_3_opcion_a    2 2 0  x \n   \nConsidera las matric...
mates_ii_integrales_2000_reserva_2_ejercicio_1_opcion_a                     Considera la función f :  definida por f(x) ...
mates_ii_integrales_2002_reserva_2_ejercicio_1_opcion_a                     a) Determina la función f :  sabiendo que f '...
mates_ii_funciones_2020_junio_ejercicio_5                                   sen(x)\nS ea f :0,2 la función definida po...
mates_ii_matrices_y_determinantes_2015_septiembre_ejercicio_3_opcion_a      Considera las matrices A= [matriz]; B = [matri...
mates_ii_integrales_2006_reserva_2_ejercicio_2_opcion_b                     Halla el área del recinto limitado por 

In [76]:
df.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv", index=False, encoding="utf-8")

In [75]:
df

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
id,,,,,,,,,,
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_a,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,Discuta la veracidad de las siguientes afirmac...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_b1,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,Dos conductores rectilíneos muy largos se disp...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_b2,Física,2025,Julio,Ejercicio B2,campo eléctrico y magnético,NaN,Una carga q\n1\n= 2  1 0 − 9 C está fija en e...,FISICA. 2025. JULIO. EJERCICIO b2,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
fisica_campo_electrico_y_magnetico_2025_junio_ejercicio_a,Física,2025,Junio,Ejercicio A,campo eléctrico y magnético,NaN,Dos partículas cargadas se mueven perpendicula...,FISICA. 2025. JUNIO. EJERCICIO a,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
fisica_campo_electrico_y_magnetico_2025_junio_ejercicio_b1,Física,2025,Junio,Ejercicio B1,campo eléctrico y magnético,NaN,El lado móvil de la espira rectangular de la f...,FISICA. 2025. JUNIO. EJERCICIO b1,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
...,...,...,...,...,...,...,...,...,...,...
quimica_acido_base_2000_reserva_3_ejercicio_3_opcion_b,Química,2000,Reserva 3,"Ejercicio 3, opción B",ácido base,NaN,En 500 mL de una disolución acuosa 0’1 M de Na...,QUÍMICA. 2000. RESERVA 3. EJERCICIO 3. OPCIÓN B,source_pdf_files/Química/ácido_base/2000 - A...,enunciados_ejercicios/Química/ácido_base/png...
quimica_acido_base_2000_reserva_3_ejercicio_5_opcion_a,Química,2000,Reserva 3,"Ejercicio 5, opción A",ácido base,NaN,a) Calcule los gramos de NaOH que se necesitan...,QUÍMICA. 2000. RESERVA 3. EJERCICIO 5. OPCIÓN A,source_pdf_files/Química/ácido_base/2000 - A...,enunciados_ejercicios/Química/ácido_base/png...
quimica_acido_base_2000_reserva_4_ejercicio_4_opcion_a,Química,2000,Reserva 4,"Ejercicio 4, opción A",ácido base,NaN,Se preparan disoluciones acuosas de las siguie...,QUÍMICA. 2000. RESERVA 4. EJERCICIO 4. OPCIÓN A,source_pdf_files/Química/ácido_base/2000 - A...,enunciados_ejercicios/Química/ácido_base/png...
